In [1]:
import json

# 1. 샘플 데이터 로드
mock_data = {
  "name": "project-alpha",
  "description": "None",
  "selectedFileContents": [
    {
      "path": "requirements.txt",
      "content": "pandas==2.0.0\nyfinance==0.2.18\nschedule==1.2.0\nrequests==2.28.2"
    },
    {
      "path": "main.py",
      "content": "import schedule\nimport time\nfrom src.collector import fetch_data\nfrom src.notifier import send_msg\n\ndef job():\n    price = fetch_data('AAPL')\n    if price > 150:\n        send_msg(f'Apple target hit: {price}')\n\nschedule.every().hour.do(job)\nwhile True:\n    schedule.run_pending()\n    time.sleep(1)"
    },
    {
      "path": "src/collector.py",
      "content": "import yfinance as yf\n\ndef fetch_data(ticker):\n    data = yf.Ticker(ticker)\n    return data.history(period='1d')['Close'].iloc[-1]"
    },
    {
      "path": "src/notifier.py",
      "content": "import requests\n\ndef send_msg(text):\n    url = f'https://api.telegram.org/bot{TOKEN}/sendMessage'\n    requests.post(url, data={'chat_id': ID, 'text': text})"
    }
  ]
}

In [5]:
from huggingface_hub import InferenceClient
import json
import os
import re
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

client = InferenceClient(
    model="Qwen/Qwen2.5-Coder-7B-Instruct",
    token=HF_TOKEN
)

def clean_response(text):
    # 따옴표 제거
    text = text.strip('"').strip("'")
    # 환각 키워드 감지
    hallucination_keywords = ["예측", "트렌드", "SMS", "email", "향후", "잠재적"]
    for keyword in hallucination_keywords:
        if keyword in text:
            print(f"⚠️ 환각 감지: '{keyword}' 포함")
    # 한자, 일본어, 러시아어 등 비한국어 문자 감지
    non_korean = re.findall(r'[\u4e00-\u9fff\u3040-\u30ff\u0400-\u04ff]', text)
    if non_korean:
        print(f"⚠️ 비정상 문자 감지: {non_korean}")
    return text

def call_ai(prompt):
    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": "당신은 코드 분석 전문가입니다. 반드시 한국어로만 작성하세요. 한자, 일본어, 중국어, 러시아어 사용 금지."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=500,
    )
    result = response.choices[0].message.content.strip()
    return clean_response(result)

def extract_readme_ai(data):
    context = json.dumps(data, indent=2, ensure_ascii=False)

    # 질문 1: 프로젝트 설명 (description 없을 때만 AI 사용)
    if not data.get("description") or data["description"] == "None":
        prompt_description = f"""아래는 GitHub 저장소의 파일 목록과 내용입니다.

[데이터]
{context}

이 프로그램을 한 줄로 설명하시오.
- 마크다운 없이 순수 텍스트 한 문장으로만 출력
- 예시: "Apple 주식 가격을 실시간으로 모니터링하여 목표가 도달 시 Telegram으로 알림을 전송하는 자동화 시스템"""
        description = call_ai(prompt_description)
    else:
        description = data["description"]

    # 질문 2: 주요 기능 추출
    prompt_features = f"""아래는 GitHub 저장소의 파일 목록과 내용입니다.

    [데이터]
    {context}

    이 프로그램의 주요 기능을 2~5가지로 작성하시오.

    [출력 규칙 - 반드시 준수]
    - 형식: "- 기능명: 설명" 한 줄로만
    - 설명은 20자 이내로 간결하게
    - 파일명, 함수명, 변수명 언급 금지
    - 사용자 관점에서 "무엇을 하는가"만 작성
    - 한국어만 사용, 한자/영어 혼용 금지
    - 추측 금지, 코드에서 확인된 것만
    - 코드에 없는 기능은 절대 작성하지 말 것
    - 확인되지 않은 기능은 목록에서 제외할 것


    [좋은 예시]
    - 주식 가격 수집: 지정한 종목의 실시간 주가를 가져옵니다.
    - 조건 알림: 목표가 초과 시 자동으로 알림을 발송합니다.

    [나쁜 예시 - 이렇게 하지 말 것]
    - collector.py의 fetch_data 함수가 yfinance库를 사용해서..."""
    features = call_ai(prompt_features)

    return {
        "description": description,
        "features": features
    }

# 실행
result = extract_readme_ai(mock_data)
print("=== 프로젝트 설명 ===")
print(result["description"])
print("\n=== 주요 기능 ===")
print(result["features"])

=== 프로젝트 설명 ===
Apple 주식 가격을 1시간마다 체크하고, 목표 가격 이상인 경우 Telegram 메시지를 보내는 자동화된 모니터링 시스템.

=== 주요 기능 ===
- 데이터 수집: 필요 패키지를 설치해 주식 데이터를 가져옵니다.
- 알림 설정: 특정 주식의 가격이 정해진 임계값을 넘으면 알림을 보냅니다.
- 시간 스케줄링: 매 시간마다 가격을 체크하고 알림을 보내도록 설정합니다.
- 실시간 감시: 무한루프를 통해 빠르게 주식 가격을 체크합니다.
